# LLM 10일차 (3) — 조건부 분기 + 비동기 + 체크포인팅 : 매뉴얼이 필요할 때만 검색하는 고객 응대 챗봇

수업 필기 `LangGraph_실습8` 을 정리 · 완성한 노트입니다. 필기는 **검색 노드 중간(`# 재구성된 질문 -> 검색 도구 실행`)에서 멈춰** 있어서, 교재 p.105 의 분기 흐름(Day 053) · 비동기(Day 054) · 체크포인팅(오늘)을 합쳐 끝까지 만들었습니다.

```
START → classifier_node ─ needs_rag=True ─→ retriever_node (재구성 + 검색) → rag_answer_node ─→ END
        (RAG / NO_RAG,     └ needs_rag=False ──────────────────────────→ general_answer_node ─┘
         대화 기록까지 보고)                       (모든 노드 async · AsyncSqliteSaver · thread_id)
```

> **출력 읽는 법** — 정리 노트북은 **API 키 없이** 만들었다.
>
> | 표시 | 뜻 |
> |---|---|
> | `# 📼` | **수업 시간에 실제로 실행한 결과**를 필기 원본에서 그대로 옮긴 출력 |
> | `# ⏳` | API 키가 필요해 **출력이 비어 있는 셀** — 직접 실행 |
> | `# 🧪` | OpenAI 대신 **로컬 임베딩** · **가짜 LLM** 으로 실제로 돌린 셀 — 답변 문장은 gpt-5-nano 와 다르다. **체크포인트 저장 · 복원 · thread_id 분리 · 오류는 진짜와 같다** |
> | 표시 없음 | 정리 노트북에서 실제로 실행한 셀 |

## 0. 설치 · 실행 방법

`pip install langgraph langgraph-checkpoint-sqlite aiosqlite langchain-openai langchain-community langchain-chroma python-dotenv` (키 없이 : `+ langchain-huggingface sentence-transformers`) · 같은 폴더에 `faq_manual.csv` · `.env` · 키가 있으면 `# ⏳`, 없으면 `# 🧪` 셀

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q langgraph-checkpoint-sqlite aiosqlite

In [2]:
import os, asyncio, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
from typing import Annotated
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel
from langchain_community.document_loaders import CSVLoader
from langchain_chroma import Chroma
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langgraph.graph import StateGraph, END, add_messages
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

loaded = load_dotenv(find_dotenv(usecwd=True), override=True)          # 키 값은 출력하지 않는다
print('.env 읽음 :', loaded, '| OPENAI_API_KEY 있음 :', bool(os.getenv('OPENAI_API_KEY')))

from typing import Literal

.env 읽음 : False | OPENAI_API_KEY 있음 : False


In [3]:
docs = CSVLoader('faq_manual.csv', encoding='utf-8', content_columns=['question', 'answer'],
                 metadata_columns=['faq_id', 'category']).load()

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
llm = ChatOpenAI(model='gpt-5-nano')
model = OpenAIEmbeddings(model='text-embedding-3-small')

In [5]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 거리는 text-embedding-3-small 과 다를 수 있다

from langchain_huggingface import HuggingFaceEmbeddings
model = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask', encode_kwargs={'normalize_embeddings': True})

In [6]:
# 🧪 LLM 대신 '정해 둔 규칙으로 답하는' 가짜 LLM — 체크포인터가 무엇을 저장 · 복원하는지 확인용

GREETING = ('안녕', '고마', '감사', '반가')

def _fake(prompt_value):
    msgs = prompt_value.to_messages()
    system, last, history = msgs[0].content, msgs[-1].content, msgs[1:-1]
    if "'RAG' 또는 'NO_RAG'" in system:                       # 의도 분류
        return AIMessage('NO_RAG' if any(w in last for w in GREETING) else 'RAG')
    if '재구성' in system:                                     # 질문 재구성 : 앞 사용자 질문 + 지금 질문 (Day 053 규칙)
        prev = [m.content for m in history if isinstance(m, HumanMessage)]
        return AIMessage(f'{prev[-1]} {last}' if prev else last)
    if 'answer: ' in system:                                   # 매뉴얼 기반 답변
        first = system.split('answer: ')[-1].split('. ')[0].strip()
        return AIMessage(f'(모의 · 기록 {len(history)}개) {first}.')
    return AIMessage(f'(모의 일반 답변 · 기록 {len(history)}개) 네, 무엇이든 물어보세요!')

llm = RunnableLambda(_fake)

In [7]:
vectorstore = Chroma.from_documents(docs, model, collection_name='faq_router')
retriever = vectorstore.as_retriever(search_kwargs={'k': 1})
print('문서 수 :', vectorstore._collection.count())

문서 수 : 10


## 1. 전용 체인 4개 (필기)

In [8]:
classifier_system_prompt = '''
당신은 온라인 교육 플랫폼 '코드크래프터스 아카데미'의 고객 문의 분류 전문가입니다.
이전 대화 기록(chat history)과 최신 사용자 질문을 함께 고려하여,
사내 고객 응대 매뉴얼(수강 신청, 회원가입, 환불 규정, 결제 수단, 강의 자료, 수료증 등) 조회가 필요한지 판단하세요.

- 플랫폼 이용, 강좌 정책, 환불, 결제, 수료증 등 회사 매뉴얼 확인이 필요한 문의(이전 매뉴얼 대화의 후속 질문 포함): 'RAG'
- 단순 인사말, 감사 표현, 일반 프로그래밍 상식/문법 질문 등 사내 매뉴얼이 필요 없는 대화: 'NO_RAG'

반드시 'RAG' 또는 'NO_RAG' 단어 하나만 정확하게 출력하세요. 다른 설명이나 공백은 일체 포함하지 마세요.
'''
classifier_chain = ChatPromptTemplate.from_messages([
    ('system', classifier_system_prompt), MessagesPlaceholder('chat_history'), ('human', '{input}')]) | llm | StrOutputParser()

rephrase_system_prompt = '''
주어진 채팅 기록(chat history)과 최신 사용자 질문이 있을 때,
이 질문이 채팅 기록의 맥락을 참조하고 있다면 모호한 질문(예: "그거 전액 다 받으려면 조건이 뭐야?")을,
채팅 기록 없이도 이해할 수 있는 명확한 질문(예: "수강 신청한 강의를 전액 환불 받으려면 조건이 뭐야?")으로 다시 작성하세요.
질문에 답하지 말고, 오직 질문을 재구성하거나 그대로 반환하기만 하세요.
'''
rephrase_chain = ChatPromptTemplate.from_messages([
    ('system', rephrase_system_prompt), MessagesPlaceholder('chat_history'), ('human', '{input}')]) | llm | StrOutputParser()

rag_qa_system_prompt = '''
당신은 코드크래프터스 아카데미의 친절한 고객 지원 담당자입니다.
아래 제공된 [매뉴얼 내용]과 이전 대화 맥락을 바탕으로 고객의 질문에 정중하고 명확하게 답변해 주세요.
매뉴얼에 명시된 내용을 왜곡하지 말고 사실에 기반하여 친절하게 안내하세요.
매뉴얼에 없는 절차나 기능(주문 조회, 고객센터 전달 등)은 약속하지 마세요.

[매뉴얼 내용]
{context}
'''
rag_qa_chain = ChatPromptTemplate.from_messages([
    ('system', rag_qa_system_prompt), MessagesPlaceholder('chat_history'), ('human', '{input}')]) | llm

general_qa_system_prompt = '''
당신은 친절하고 명쾌한 답변을 하는 AI 비서입니다.
이전 대화 맥락을 기억하며 사용자의 질문이나 인사에 자연스럽고 유익하게 답변해 주세요.
'''
general_qa_chain = ChatPromptTemplate.from_messages([
    ('system', general_qa_system_prompt), MessagesPlaceholder('chat_history'), ('human', '{input}')]) | llm

> **📝 고친 부분** 필기 주석 `이전 대화 매락` → 맥락 · `블리언` → 불리언 · 재구성 프롬프트의 조사 빠짐('모호한 질문(예: …)**을**'). RAG 답변 프롬프트에는 Day 054 수업 출력에서 본 '없는 기능 약속' 을 막는 줄을 더했다.
> 분류 프롬프트가 **대화 기록까지** 본다는 점이 Day 053 분기판과 다르다 — '그거 전액은?' 같은 후속 질문은 단어만 보면 매뉴얼 질문인지 알기 어렵기 때문이다(프롬프트의 '이전 매뉴얼 대화의 후속 질문 포함').

## 2. 상태 · 노드 (필기 + 멈춘 곳부터)

In [9]:
class AgentState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = []
    needs_rag: bool = False
    context: str = ''

def parse_decision(text: str) -> bool:
    # Day 052 : 따옴표 · 마침표를 떼고 비교 ('RAG' in 'NO_RAG' 함정도 피한다)
    return text.strip().strip('\'"`.').strip().upper() == 'RAG'

async def classifier_node(state: AgentState) -> dict:
    history, question = state.messages[:-1], state.messages[-1].content
    decision = await classifier_chain.ainvoke({'chat_history': history, 'input': question})
    needs_rag = parse_decision(decision)
    print(f'  [1 분류] {question!r} → {decision} (needs_rag={needs_rag})')
    return {'needs_rag': needs_rag, 'context': ''}          # 📝 매 턴 context 를 비운다 (앞 턴 매뉴얼이 남지 않게)

async def retriever_node(state: AgentState) -> dict:
    history, question = state.messages[:-1], state.messages[-1].content
    query = await rephrase_chain.ainvoke({'chat_history': history, 'input': question}) if history else question
    found = await retriever.ainvoke(query)                   # ← 필기가 멈춘 곳 : 재구성된 질문으로 검색
    context = '\n\n'.join(d.page_content for d in found)
    print(f"  [2 검색] 재구성 : {query} → {[d.metadata['faq_id'] for d in found]}")
    return {'context': context}

async def rag_answer_node(state: AgentState) -> dict:
    response = await rag_qa_chain.ainvoke({'context': state.context, 'chat_history': state.messages[:-1],
                                           'input': state.messages[-1].content})
    print(f'  [3 매뉴얼 답변] {response.content[:60]}')
    return {'messages': [response]}

async def general_answer_node(state: AgentState) -> dict:
    response = await general_qa_chain.ainvoke({'chat_history': state.messages[:-1], 'input': state.messages[-1].content})
    print(f'  [3 일반 답변] {response.content[:60]}')
    return {'messages': [response]}

def route_by_rag_need(state: AgentState) -> Literal['retriever_node', 'general_answer_node']:
    return 'retriever_node' if state.needs_rag else 'general_answer_node'

## 3. 그래프

In [10]:
workflow = StateGraph(AgentState)
workflow.add_node('classifier_node', classifier_node)
workflow.add_node('retriever_node', retriever_node)
workflow.add_node('rag_answer_node', rag_answer_node)
workflow.add_node('general_answer_node', general_answer_node)
workflow.set_entry_point('classifier_node')
workflow.add_conditional_edges('classifier_node', route_by_rag_need,
                               {'retriever_node': 'retriever_node', 'general_answer_node': 'general_answer_node'})
workflow.add_edge('retriever_node', 'rag_answer_node')
workflow.add_edge('rag_answer_node', END)
workflow.add_edge('general_answer_node', END)
print(workflow.compile().get_graph().draw_mermaid().split('graph TD;')[1].split('classDef')[0])


	__start__([<p>__start__</p>]):::first
	classifier_node(classifier_node)
	retriever_node(retriever_node)
	rag_answer_node(rag_answer_node)
	general_answer_node(general_answer_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> classifier_node;
	classifier_node -.-> general_answer_node;
	classifier_node -.-> retriever_node;
	retriever_node --> rag_answer_node;
	general_answer_node --> __end__;
	rag_answer_node --> __end__;
	


## 4. 실행 — 한 대화방에서 4턴

In [11]:
# 🧪 LLM 대신 '정해 둔 규칙으로 답하는' 가짜 LLM — 체크포인터가 무엇을 저장 · 복원하는지 확인용
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 거리는 text-embedding-3-small 과 다를 수 있다

db_file = 'router_agent_demo.sqlite'
if os.path.exists(db_file):
    os.remove(db_file)

async def main(questions, thread_id='user-8888'):
    async with AsyncSqliteSaver.from_conn_string(db_file) as memory:
        agent = workflow.compile(checkpointer=memory)
        config = {'configurable': {'thread_id': thread_id}}
        for user_input in questions:
            print(f'사용자: {user_input}')
            path = []
            async for update in agent.astream({'messages': [HumanMessage(user_input)]}, config=config, stream_mode='updates'):
                path.append(next(iter(update)))
            state = (await agent.aget_state(config)).values
            print(f"  경로 : {' → '.join(path)} | messages {len(state['messages'])}개 | context {len(state['context'])}자\n")

await main(['안녕하세요!', '수강 신청한 강의를 환불하고 싶어요', '그거 전액 다 받으려면 조건이 어떻게 되나요?', '고마워요'])

사용자: 안녕하세요!
  [1 분류] '안녕하세요!' → NO_RAG (needs_rag=False)
  [3 일반 답변] (모의 일반 답변 · 기록 0개) 네, 무엇이든 물어보세요!
  경로 : classifier_node → general_answer_node | messages 2개 | context 0자

사용자: 수강 신청한 강의를 환불하고 싶어요
  [1 분류] '수강 신청한 강의를 환불하고 싶어요' → RAG (needs_rag=True)
  [2 검색] 재구성 : 안녕하세요! 수강 신청한 강의를 환불하고 싶어요 → ['FAQ-002']
  [3 매뉴얼 답변] (모의 · 기록 2개) 네, 환불 가능합니다.
  경로 : classifier_node → retriever_node → rag_answer_node | messages 4개 | context 152자

사용자: 그거 전액 다 받으려면 조건이 어떻게 되나요?
  [1 분류] '그거 전액 다 받으려면 조건이 어떻게 되나요?' → RAG (needs_rag=True)
  [2 검색] 재구성 : 수강 신청한 강의를 환불하고 싶어요 그거 전액 다 받으려면 조건이 어떻게 되나요? → ['FAQ-002']
  [3 매뉴얼 답변] (모의 · 기록 4개) 네, 환불 가능합니다.
  경로 : classifier_node → retriever_node → rag_answer_node | messages 6개 | context 152자

사용자: 고마워요
  [1 분류] '고마워요' → NO_RAG (needs_rag=False)
  [3 일반 답변] (모의 일반 답변 · 기록 6개) 네, 무엇이든 물어보세요!
  경로 : classifier_node → general_answer_node | messages 8개 | context 0자



- 인사 · 감사는 `classifier → general_answer`, 매뉴얼 질문은 `classifier → retriever → rag_answer` 로 갔다 — 매 턴 **새 질문 하나만** 넣었고 기록은 체크포인터가 복원했다
- 3턴 '그거 전액 …' 은 재구성 후 환불 FAQ(FAQ-002)
- 4턴 '고마워요' 뒤 `context` 가 **0자** — 분류 노드가 매 턴 비우기 때문이다. 비우지 않으면 3턴의 환불 매뉴얼이 그대로 DB 상태에 남는다(SQLite 노트 §4)
- `astream(…, stream_mode='updates')` 도 `ainvoke` 처럼 `config` 를 받는다 — 노드 경로를 보면서 체크포인트도 저장된다

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

# 필기 main() 그대로의 대화 루프 (직접 입력)
async def chat():
    async with AsyncSqliteSaver.from_conn_string('router_agent.sqlite') as memory:
        agent = workflow.compile(checkpointer=memory)
        config = {'configurable': {'thread_id': 'user-8888'}}
        while True:
            user_input = input('사용자: ')
            if user_input.strip().lower() == 'exit':
                break
            final_state = await agent.ainvoke({'messages': [HumanMessage(content=user_input)]}, config=config)
            print(f"AI 에이전트:\n{final_state['messages'][-1].content}\n" + '-' * 80)

await chat()

## 정리 — 오늘까지 쌓은 것

| 기능 | 배운 날 | 이 그래프에서 |
|---|---|---|
| 조건부 엣지 | Day 052 | `add_conditional_edges('classifier_node', route_by_rag_need, {...})` |
| RAG 노드 · 재구성 · 대화 기록 경로 | Day 053 | `retriever_node` 안에서 재구성 → 검색 · `MessagesPlaceholder('chat_history')` |
| 비동기 | Day 054 | 노드 4개 `async def` · `await … ainvoke` · `await main()` |
| 체크포인팅 | Day 055 | `AsyncSqliteSaver` + `thread_id` — 새 질문만 넣으면 기록 복원 |
| 📝 상태 위생 | Day 055 | 리듀서 없는 `context` 는 매 턴 비우기 · 같은 thread_id 는 다시 켜도 이어짐 |